# Ejercicio: Cúmulo globular M15
Messier 15 (M15), también conocido como NGC 7078, es un cúmulo globular situado en la constelación de Pegaso. Con una edad estimada de 12.000 millones de años, es uno de los cúmulos globulares más antiguos conocidos y una reliquia de los primeros años de la Vía Láctea.
ADQL con Python- Obtendremos los datos de estrellas provenientes de Gaia DR3, para este cúmulo.



1. Buscar las coordenadas del cúmulo globular M15 desde Simbad.
2. Crear una 'query' para obtener los siguientes parámetros para M2 desde Gaia DR3: source id, ra, dec, parallax, magnitude en G, movimientos propios en RA y Dec, y sus errores, color bp-rp, y ruwe. Utilice un radio de búsqueda de 5 arcmin.
3. Guardar el catálogo como 'catM15_1.txt'
4. Repita la 'query' pero considerando las siguientes condiciones: Valores de ruwe menores a 1.5 Magnitudes en G más brillantes o iguales que 19 mag. Errores en movimientos propios menores a 0.5 en RA y Dec.
5. Guardar el catálogo como 'catM15_2.txt'
6. A la tabla resultante de la segunda 'query', agregar una nueva columna de nombre PM tot (total proper motion) correspondiente a la magnitud del movimiento propio total considerando sus componentes en RA y Dec.
7. Guardar el catalogo como 'catM15_3.txt' con las columnas ordenadas de la siguiente manera: source id, ra, movimiento propio en RA, Dec, movimiento propio en Dec, magnitud en G y color bp-rp.
8. Construya un plot para el movimiento propio de las estrellas del cúmulo del catálogo 1 (sin restricciones), y en rojo las estrellas del catálogo 2.
9. Construya un diagrama color-magnitud para el cúmulo para todas las estrellas del catálogo 1 (sin restricciones), y en rojo las estrellas del catálogo 2.


## PREGUNTAS ##

1. ¿Cuál es el centro exacto de M2 y qué tipo de objeto es según Simbad?

2. ¿Por qué es importante obtener estas coordenadas con precisión antes de consultar Gaia?

3. ¿Qué tipos de estrellas (miembros del cúmulo, estrellas del fondo galáctico, etc.) podrían estar incluidas en la muestra inicial?

4. ¿Qué representa el parámetro ruwe y por qué es útil para filtrar datos astrométricos?

6. ¿Qué impacto esperas que tenga este filtrado sobre el número total de estrellas en la muestra?

7. ¿Qué información física nos entrega el módulo del vector de movimiento propio (PM tot)?

8. ¿Qué patrón se observa en el gráfico de pmRA vs. pmDec para las estrellas del cúmulo?

9. ¿Cómo se diferencia el grupo más concentrado (miembros del cúmulo) del fondo galáctico en ese gráfico?

10. ¿Qué representa el grupo disperso de puntos que no sigue la tendencia principal?

11. ¿Puedes identificar estructuras como la secuencia principal, la rama de gigante roja o la rama horizontal en el diagrama color-magnitud del cúmulo?

12. ¿Qué diferencias observas entre el catálogo sin filtros y el catálogo con filtros aplicados?

13. ¿Cómo cambiarían los resultados si aumentaras el radio de búsqueda?

In [ ]:
!pip install astroquery

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.2/910.2 kB 42.4 MB/s eta 0:00:00


In [ ]:
from astroquery.gaia import Gaia

#Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"  # Data Release 3 por defecto

# Si necesita otro de los anteriore, entonces:
# Gaia.MAIN_GAIA_TABLE = "gaiadr2.gaia_source"  # Select Data Release 2


 "IMPORTANT: The GACS archive will be down from April 18, 2024, 16:30 CEST until April 23, 2024, 08:30 CEST."


In [ ]:
# Buscar coordenadas de un objeto en Simbad

from astroquery.simbad import Simbad

object_table = Simbad.query_object("M15")
object_table

MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,int32
M 15,21 29 58.33,+12 10 01.2,6,6,--,--,0,D,O,2010AJ....140.1830G,1


In [ ]:
#Obtener coordenadas del objeto en grados

from astroquery.gaia import Gaia
import astropy.units as u
from astropy.coordinates import SkyCoord

coord = SkyCoord(ra = object_table['RA'], dec = object_table['DEC'], unit=(u.hourangle, u.deg))

reff = 1 * u.arcmin   # radio efectivo del cúmulo globular
n = 1
search_radius = n * reff
search_radius = reff.to("deg")

print(coord.ra, coord.dec)
print(coord.ra.value, coord.dec.value)


[322d29m34.95s] [12d10m01.2s]
[322.49304167] [12.167]


In [ ]:
# Definir una "Query"

query = "SELECT source_id, ra, dec, phot_g_mean_mag, ruwe \
        FROM gaiadr3.gaia_source \
        WHERE CONTAINS(POINT('ICRS',gaiadr3.gaia_source.ra,gaiadr3.gaia_source.dec), CIRCLE('ICRS',\
        %.8f,%.8f,%.8f)"%(coord.ra.value, coord.dec.value, search_radius.value) + ")=1"


# Arroja una cantidad ilimitada de filas.
Gaia.ROW_LIMIT = -1


job     = Gaia.launch_job_async(query)
results = job.get_results()

removejob = Gaia.remove_jobs([job.jobid])



INFO: Query finished. [astroquery.utils.tap.core]
Removed jobs: '['1681836747915O']'.


In [ ]:
print(results)


     source_id              ra         ... phot_g_mean_mag    ruwe  
                           deg         ...       mag                
------------------- ------------------ ... --------------- ---------
6760426206386378496 283.76620117674923 ...       16.559338 0.9074039
6760426206386378624  283.7662914797505 ...       17.064053        --
6760426137666900864   283.776500235391 ...        18.93016 1.4429663
6760426137666920320  283.7753319833414 ...       18.505827        --
6760426137666920576   283.775263131314 ...       18.406166        --
6760426137666921088  283.7774281240007 ...        19.08253        --
6760426137666921216  283.7777899989195 ...        19.54066        --
6760426137666922496 283.77972088050393 ...       18.577139  1.188274
6760426137666922880  283.7785599846743 ...       16.983522 1.2813452
6760426137666970368  283.7724495819428 ...       17.212385 1.2237955
                ...                ... ...             ...       ...
6760429161324126848  283.755865004

In [ ]:
# Definir una "Query"

query = "SELECT source_id, ra, dec, parallax, phot_g_mean_mag AS gmag, \
        pmra, pmra_error AS e_pmra, \
        pmdec, pmdec_error AS e_pmdec, \
        bp_rp, ruwe \
        FROM gaiadr3.gaia_source \
        WHERE CONTAINS(POINT('ICRS',gaiadr3.gaia_source.ra,gaiadr3.gaia_source.dec), CIRCLE('ICRS',\
        %.8f,%.8f,%.8f)"%(coord.ra.value, coord.dec.value, search_radius.value) + ")=1"


# Arroja una cantidad ilimitada de filas.
Gaia.ROW_LIMIT = -1


job     = Gaia.launch_job_async(query)
results = job.get_results()

removejob = Gaia.remove_jobs([job.jobid])

INFO: Query finished. [astroquery.utils.tap.core]
Removed jobs: '['1681839121165O']'.


In [ ]:
print(results["gmag"])

   gmag  
   mag   
---------
16.559338
17.064053
 18.93016
18.505827
18.406166
 19.08253
 19.54066
18.577139
16.983522
17.212385
      ...
19.474928
19.946959
20.119476
20.199963
19.809011
20.108616
 20.16699
 18.03828
19.526587
18.384798
Length = 2232 rows
